In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings("ignore")

In [3]:
!pip install dagshub mlflow
import dagshub
dagshub.init(repo_owner='smama23', repo_name='MLassignment2', mlflow=True)

import mlflow
mlflow.set_experiment("AdaBoost_Training")
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 6.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=dbbe5eb0-6732-4f87-8a37-474353275b2e&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b41fccb7a72ccd677045ad8c638cb45286790ce9dbe10b5898bf0e384aef6a9f




Accessing as smama23

Initialized MLflow to track repo "smama23/MLassignment2"

Repository smama23/MLassignment2 initialized!

🏃 View run clumsy-skunk-353 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4/runs/f63cd58198ba4075948e371e94900191
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4


In [4]:
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")

Cleaning

In [5]:
with mlflow.start_run(run_name="AdaBoost_Cleaning_v1"):
    
    df_clean = df.copy()
    
    y = df_clean["isFraud"]
    df_clean = df_clean.drop(columns=["isFraud"])
    
    df_clean = df_clean.fillna(-999)
    
    mlflow.log_param("cleaning", "fillna_-999")
    mlflow.log_metric("num_features", df_clean.shape[1])

🏃 View run AdaBoost_Cleaning_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4/runs/bd40e9aee4454f3db25a87842e3331c8
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4


Feature Engineering

In [6]:
with mlflow.start_run(run_name="AdaBoost_FeatureEngineering_v1"):
    
    df_fe = df_clean.copy()
    
    df_fe["hour"] = (df_fe["TransactionDT"] // 3600) % 24
    df_fe["day"] = (df_fe["TransactionDT"] // (3600 * 24)) % 7
    df_fe["log_amount"] = np.log1p(df_fe["TransactionAmt"])
    
    mlflow.log_param("features_added", "hour, day, log_amount")
    mlflow.log_metric("num_features_after_fe", df_fe.shape[1])

🏃 View run AdaBoost_FeatureEngineering_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4/runs/632e629bf4db45e5a461cb9b1e1c79eb
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4


In [7]:
def frequency_encoding(df, cols):
    df = df.copy()
    for col in cols:
        freq = df[col].value_counts() / len(df)
        df[col] = df[col].map(freq)
    return df

Feature Selection

In [8]:
with mlflow.start_run(run_name="AdaBoost_FeatureSelection_v1"):
    
    df_enc = df_fe.copy()
    
    cat_cols = df_enc.select_dtypes(include="object").columns
    df_enc = frequency_encoding(df_enc, cat_cols)
    
    nunique = df_enc.nunique()
    df_fs = df_enc[nunique[nunique > 1].index]
    
    mlflow.log_param("encoding", "frequency")
    mlflow.log_param("feature_selection", "remove_constant")
    mlflow.log_metric("num_features_final", df_fs.shape[1])

🏃 View run AdaBoost_FeatureSelection_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4/runs/c737fdf8892c4cbbaad6337e3c90d881
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4


Training

In [9]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [10]:
if mlflow.active_run():
    mlflow.end_run()

with mlflow.start_run(run_name="AdaBoost_Training_v1"):

    X_train, X_val, y_train, y_val = train_test_split(
        df_fs, y, test_size=0.2, stratify=y, random_state=42
    )
    
    model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=50,
        learning_rate=1.0,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("n_estimators", 50)
    mlflow.log_param("base_depth", 1)
    
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(model, "model")
    
    print(train_auc, val_auc)

2026/05/04 09:26:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:27:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.8577682791848662 0.8563850777188212
🏃 View run AdaBoost_Training_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4/runs/caa2af10027e421980861a662fc67440
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4


In [11]:
import gc
import numpy as np

if mlflow.active_run():
    mlflow.end_run()

def take_rows(X, idx):
    return X.iloc[idx] if hasattr(X, "iloc") else X[idx]

train_sample_size = min(30000, len(X_train))

rng = np.random.default_rng(42)
train_idx = rng.choice(len(X_train), train_sample_size, replace=False)

X_train_eval = take_rows(X_train, train_idx)
y_train_eval = take_rows(y_train, train_idx)

with mlflow.start_run(run_name="AdaBoost_Training_v2"):
    
    model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1,
            min_samples_leaf=50,
            random_state=42
        ),
        n_estimators=50,
        learning_rate=0.5,
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict_proba(X_train_eval)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train_eval, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_params({
        "n_estimators": 50,
        "base_depth": 1,
        "learning_rate": 0.5,
        "min_samples_leaf": 50,
        "train_auc_sample_size": train_sample_size
    })
    
    mlflow.log_metrics({
        "train_auc_sampled": train_auc,
        "val_auc": val_auc
    })
    
    mlflow.set_tag("model_logged", "false")
    
    print(train_auc, val_auc)
    
    del y_train_pred
    del y_val_pred
    gc.collect()

0.8661360844527051 0.8519330495293395
🏃 View run AdaBoost_Training_v2 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4/runs/4cf77b6709774c52b4ef47381bf2ed82
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4


In [12]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in self.cols:
            freq = X[col].value_counts() / len(X)
            self.freq_maps[col] = freq
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            X[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X


class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["hour"] = X["TransactionDT"]
        X["day"] = X["TransactionDT"]
        X["log_amount"] = np.log1p(X["TransactionAmt"])
        return X


class FillNaTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, fill_value=-999):
        self.fill_value = fill_value

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.fillna(self.fill_value)

In [13]:
cat_cols = df.drop(columns=["isFraud"]).select_dtypes(include="object").columns

In [17]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("cleaning", FillNaTransformer()),
    ("feature_engineering", FeatureEngineer()),
    ("encoding", FrequencyEncoder(cat_cols)),
    ("model", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=50,
        learning_rate=1.0,
        random_state=42
    ))
])

In [18]:
mlflow.end_run(status="FINISHED")

with mlflow.start_run(run_name="AdaBoost_Pipeline_v1"):
    
    X = df.drop(columns=["isFraud"])
    y = df["isFraud"]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    
    pipeline.fit(X_train, y_train)
    
    y_train_pred = pipeline.predict_proba(X_train)[:, 1]
    y_val_pred = pipeline.predict_proba(X_val)[:, 1]
    
    train_auc = roc_auc_score(y_train, y_train_pred)
    val_auc = roc_auc_score(y_val, y_val_pred)
    
    mlflow.log_param("model", "AdaBoost_pipeline")
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    
    mlflow.sklearn.log_model(pipeline, "rf_pipeline_model")
    
    print(train_auc, val_auc)

2026/05/04 09:37:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 09:37:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.8577682791848662 0.8563850777188212
🏃 View run AdaBoost_Pipeline_v1 at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4/runs/21dc6b17f4e84fd19a091caab5533cef
🧪 View experiment at: https://dagshub.com/smama23/MLassignment2.mlflow/#/experiments/4
